# My Spain Top 50 Songs Project
In this notebook, I process my playlist data to calculate some cool stats!

In [ ]:
import pandas as pd
import numpy as np
import datetime
import json

## Loading Up the Data

In [ ]:
df = pd.read_csv('/Users/amitrao/Desktop/internship 2nd project/Atlantic_Spain.csv')
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
df['song'] = df['song'].astype(str).str.strip()
df['artist'] = df['artist'].astype(str).str.strip()

entries_per_day = df.groupby('date').size()
if not all(entries_per_day == 50):
    print('Warning: Not all days have exactly 50 entries.')

df = df.sort_values(by=['date', 'position'])
df['song_id'] = df['song'] + ' - ' + df['artist']
df.head()

## Finding Out How Long Songs Last (Lifecycles)

In [ ]:
first_appearance = df.groupby('song_id')['date'].min().reset_index().rename(columns={'date': 'first_appearance'})
last_appearance = df.groupby('song_id')['date'].max().reset_index().rename(columns={'date': 'last_appearance'})
df = df.merge(first_appearance, on='song_id').merge(last_appearance, on='song_id')

lifecycle = df.groupby('song_id').agg(
    entry_date=('date', 'min'),
    exit_date=('date', 'max'),
    total_days=('date', 'count'),
    peak_position=('position', 'min'),
    is_explicit=('is_explicit', 'first'),
    album_type=('album_type', 'first'),
    duration_ms=('duration_ms', 'mean'),
    popularity_mean=('popularity', 'mean'),
    popularity_peak=('popularity', 'max')
).reset_index()

peak_dates = df.loc[df.groupby('song_id')['position'].idxmin(), ['song_id', 'date']].rename(columns={'date': 'peak_date'})
peak_dates = peak_dates.drop_duplicates(subset=['song_id'])
lifecycle = lifecycle.merge(peak_dates, on='song_id')
lifecycle['time_to_peak'] = (lifecycle['peak_date'] - lifecycle['entry_date']).dt.days
lifecycle.head()

## Sorting Songs into Phases

In [ ]:
df['days_since_entry'] = (df['date'] - df['first_appearance']).dt.days
df['days_to_exit'] = (df['last_appearance'] - df['date']).dt.days

def classify_stage(row):
    if row['days_since_entry'] <= 7:
        return 'New Entry'
    if row['days_to_exit'] <= 7 and row['total_days'] > 14:
        return 'Decline Phase'
    if row['position'] <= 10:
        return 'Peak Phase'
    return 'Mature Phase'

df = df.merge(lifecycle[['song_id', 'total_days']], on='song_id')
df['lifecycle_stage'] = df.apply(classify_stage, axis=1)
df = df.drop(columns=['first_appearance', 'last_appearance', 'total_days'])
df['lifecycle_stage'].value_counts()

## Calculating the Final Stats!

In [ ]:
daily_new_entries = df[df['days_since_entry'] == 0].groupby('date').size()
all_dates = df['date'].unique()
daily_new_entries = daily_new_entries.reindex(all_dates, fill_value=0)
avg_churn_rate = daily_new_entries.mean() / 50.0

avg_days = lifecycle['total_days'].mean()
avg_time_to_peak = lifecycle['time_to_peak'].mean()

explicit_days = lifecycle[lifecycle['is_explicit'] == True]['total_days'].mean()
clean_days = lifecycle[lifecycle['is_explicit'] == False]['total_days'].mean()
explicit_score = explicit_days / clean_days if clean_days else 0

single_days = lifecycle[lifecycle['album_type'] == 'single']['total_days'].mean()
album_days = lifecycle[lifecycle['album_type'] == 'album']['total_days'].mean()
single_vs_album_ratio = single_days / album_days if album_days else 0

retention_stability_index = lifecycle[lifecycle['total_days'] > 30].shape[0] / lifecycle.shape[0]

results = {
    'Average Days on Playlist': float(avg_days),
    'Entry-to-Peak Time (days)': float(avg_time_to_peak),
    'Average Daily Playist Churn (%)': float(avg_churn_rate * 100),
    'Retention Stability Index (>30 days %)': float(retention_stability_index * 100),
    'Explicit Content Lifecycle Score': float(explicit_score),
    'Single vs Album Longevity Ratio': float(single_vs_album_ratio)
}

print(json.dumps(results, indent=4))

with open('/Users/amitrao/Desktop/internship 2nd project/kpi_results.json', 'w') as f:
    json.dump(results, f, indent=4)
df.to_csv('/Users/amitrao/Desktop/internship 2nd project/processed_spain_top50.csv', index=False)